In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/extensions.py:151: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/annotated/checkpoints/post_cell_10.pickle

[[<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f95ab652d40>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948dc4ab30>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948dc4b730>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948dc49f90>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948dc4b5e0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948dc4b1f0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948dc4a830>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948dc4b610>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f95ab651d50>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948f6c7460>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948dc4a890>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948dc

me:  11
trying: ['null_rate']
me:  4
trying: ['mf_ratio']
me:  9
trying: ['r']
me:  9
trying: ['i_1']
me:  10


Declaring variable pd
Declaring variable np
Declaring variable time
Declaring variable dt
Declaring variable warnings
Declaring variable factor
Declaring variable i
Declaring variable null_rate
Declaring variable x
Declaring variable y
Declaring variable mf_ratio
Declaring variable r
Declaring variable i_2
Declaring variable i_1
Declaring variable ratings_ages
Declaring variable df
Declaring variable orig_output
Declaring variable data


In [4]:
%%RecordEvent
%%time
### cell 11 ###

country_order = df['first_country'].value_counts()[:11].index

# Use GPU‐friendly pivot instead of CPU‐bound value_counts + unstack
counts = df.groupby(['first_country', 'type']).size().reset_index(name='count')
data_q2q3 = (
    counts
    .pivot(index='first_country', columns='type', values='count')
    .fillna(0)
    .loc[country_order]
)

# compute total and ratios
data_q2q3['sum'] = data_q2q3.sum(axis=1)
data_q2q3_ratio = (
    data_q2q3[['Movie', 'TV Show']]
    .div(data_q2q3['sum'], axis=0)
    .sort_values(by='Movie', ascending=False)
    .iloc[::-1]
)

CPU times: user 284 ms, sys: 81.6 ms, total: 365 ms
Wall time: 400 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high/checkpoints/post_cell_11_try_1.pickle

migration speed (bps): 795704815.479484
---------------------------
variables to migrate:
i_1 53
np 72
time 72
country_order 48
dt 72
warnings 72
x 48
ratings_ages 640
df 48
orig_output 16
data_q2q3_ratio 48
data 48
i 60
data_q2q3 48
null_rate 32
pd 72
counts 48
factor 28
y 28
r 48
mf_ratio 48
i_2 53
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high/checkpoints/post_cell_11_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['df']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['df']
Active variables ['df']
Intermediate variables ['factor']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables ['df']
Active variables []
Intermediate variables ['null_rate', 'i']
Future variables ['df']
Modified dataframes
======= Cell 3 =======
Input variables ['df']
Active variables ['df']
Intermediate variables []
Future variables []
Modified dataframes
  df
    Input columns: set()
    Changed columns: {'description', 'country', 'director', 'show_id', 'duration', 'release_year', 'cast', 'listed_in', 'title', 'date_added', 'type', 'rating'}
    Created columns: set()
    Deleted columns: set()
======= Cell 4 =======
Input variables ['df']
Active variables []
Intermediate variables []
Future variables ['df']
Modified dataframes
======= Cell 5 =======
Input variables ['df']
Active va

In [7]:
with open(
    "/home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/opt_cell_exec_info_11_try_1.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[11], f)

In [8]:
opt_output = Out.get(4)

In [9]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/annotated/checkpoints/post_cell_11.pickle

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

[[<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c69b0a0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c719e40>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c719480>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c719de0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c7192d0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c719300>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c7183d0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c719e10>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c71a080>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c718430>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c7191b0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f948c7

me:  11
trying: ['orig_output']
me:  12
trying: ['data']
me:  13
trying: ['i']
me:  4
trying: ['null_rate']
me:  4
trying: ['pd']
me:  0
trying: ['factor']
me:  3
trying: ['y']
me:  9
trying: ['r']
me:  9
trying: ['mf_ratio']
me:  9
trying: ['i_2']
me:  10
trying: ['i_1']
me:  10


Declaring variable np
Declaring variable time
Declaring variable pd
Declaring variable dt
Declaring variable warnings
Declaring variable factor
Declaring variable i
Declaring variable null_rate
Declaring variable x
Declaring variable y
Declaring variable r
Declaring variable mf_ratio
Declaring variable i_2
Declaring variable i_1
Declaring variable ratings_ages
Declaring variable df
Declaring variable orig_output
Declaring variable data
Declaring variable country_order
Declaring variable data_q2q3
Declaring variable data_q2q3_ratio
